# Distillation Column Model Training


In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBRegressor, XGBClassifier


In [2]:
telemetry = pd.read_csv('column_telemetry.csv')
labels = pd.read_csv('column_labels.csv')
df = telemetry.merge(labels, on=['timestamp','asset_id'])


In [3]:
FEATURES = [
'feed_flow_rate',
'column_pressure',
'top_temperature',
'bottom_temperature',
'reflux_ratio',
'tray_efficiency',
'product_purity'
]
X = df[FEATURES]


In [4]:
# RUL Model
y_rul = df['rul_days']
X_train,X_test,y_train,y_test = train_test_split(X,y_rul,test_size=0.2,random_state=42)
rul_model = XGBRegressor(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
rul_model.fit(X_train,y_train)
pred_rul = rul_model.predict(X_test)
mae = mean_absolute_error(y_test,pred_rul)
rmse = np.sqrt(mean_squared_error(y_test,pred_rul))
r2 = r2_score(y_test,pred_rul)
joblib.dump(rul_model,'column_rul_model.pkl')


['column_rul_model.pkl']

In [5]:
# Failure Probability Model
y_fail = df['failure_next_30_days']
X_train,X_test,y_train,y_test = train_test_split(X,y_fail,test_size=0.2,random_state=42)
failure_model = XGBClassifier(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
failure_model.fit(X_train,y_train)
pred_fail = failure_model.predict(X_test)
acc = accuracy_score(y_test,pred_fail)
prec = precision_score(y_test,pred_fail)
rec = recall_score(y_test,pred_fail)
f1 = f1_score(y_test,pred_fail)
joblib.dump(failure_model,'column_failure_probability_model.pkl')


['column_failure_probability_model.pkl']

In [6]:
# Failure Mode Model
encoder = LabelEncoder()
y_mode = encoder.fit_transform(df['failure_mode'])
X_train,X_test,y_train,y_test = train_test_split(X,y_mode,test_size=0.2,random_state=42)
mode_model = XGBClassifier(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
mode_model.fit(X_train,y_train)
pred_mode = mode_model.predict(X_test)
mode_acc = accuracy_score(y_test,pred_mode)
joblib.dump(mode_model,'column_failure_mode_model.pkl')
joblib.dump(encoder,'column_failure_mode_encoder.pkl')


['column_failure_mode_encoder.pkl']

In [7]:
pd.DataFrame({'Metric':['MAE','RMSE','R2'],'Value':[mae,rmse,r2]}).to_csv('column_rul_metrics.csv',index=False)
pd.DataFrame({'Metric':['Accuracy','Precision','Recall','F1'],'Value':[acc,prec,rec,f1]}).to_csv('column_failure_probability_metrics.csv',index=False)
pd.DataFrame({'Metric':['Accuracy'],'Value':[mode_acc]}).to_csv('column_failure_mode_metrics.csv',index=False)
pd.DataFrame({'Feature':FEATURES,'Importance':rul_model.feature_importances_}).to_csv('column_rul_feature_importance.csv',index=False)
print('Distillation Column models and metrics saved successfully')


Distillation Column models and metrics saved successfully
